# 21 - Custom Data Pipeline Demo

Ingest instructor-provided custom `.txt`, `.csv`, or `.jsonl` documents, normalize them into the project retrieval schema, build a dense/BM25 index, and run a retrieval smoke test. This satisfies the custom document collection requirement.

In [ ]:
!pip install -q -U "sentence-transformers>=3.0.0" transformers accelerate faiss-cpu rank-bm25 pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

custom_input_dir = DRIVE_ROOT / 'data/custom_docs'
custom_output_csv = DRIVE_ROOT / 'data/processed/custom_corpus_v1.csv'
custom_output_jsonl = DRIVE_ROOT / 'data/processed/custom_corpus_v1.jsonl'
custom_report_json = DRIVE_ROOT / 'reports/custom_ingestion_report_v1.json'
custom_index_root = DRIVE_ROOT / 'indexes/custom_v1'

embedding_model = 'Qwen/Qwen3-Embedding-8B'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

custom_input_dir.mkdir(parents=True, exist_ok=True)
print('Put instructor files here:', custom_input_dir)
print('Device:', device)

In [ ]:
# Optional: create a tiny sample file if the folder is empty.
# Delete this sample or replace it with instructor documents for the real demo.
if not any(p.is_file() and p.suffix.lower() in {'.txt', '.csv', '.jsonl'} for p in custom_input_dir.rglob('*')):
    sample_path = custom_input_dir / 'sample_custom_law_note.txt'
    sample_path.write_text(
        'Örnek özel doküman: Kira uyuşmazlıklarında başvuru yolları.\n\n'
        'Kirac? kira bedelini ?demezse, s?zle?me ve ilgili mevzuat h?k?mleri ?er?evesinde yaz?l? bildirim, ?deme s?resi ve dava/takip yollar? de?erlendirilebilir.\n\n'
        'Bu dosya yaln?zca custom pipeline demosu i?indir; resmi mevzuat kayna?? de?ildir.\n',
        encoding='utf-8',
    )
    print('Sample custom file created:', sample_path)
else:
    print('Custom files found:')
    for path in sorted(custom_input_dir.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.txt', '.csv', '.jsonl'}:
            print('-', path)

In [ ]:
from src.ingest_custom_documents import ingest_custom_documents

report = ingest_custom_documents(
    input_dir=custom_input_dir,
    output_csv=custom_output_csv,
    output_jsonl=custom_output_jsonl,
    report_json=custom_report_json,
    max_chars=1800,
    overlap_chars=180,
    min_chars=30,
)
report

In [ ]:
import pandas as pd

df = pd.read_csv(custom_output_csv, dtype=str, keep_default_na=False)
print(df.shape)
df[['record_id', 'doc_key', 'article_key', 'citation_label', 'retrieval_text']].head(5)

In [ ]:
from src.build_index import build_indexes

manifest = build_indexes(
    corpus_path=custom_output_csv,
    index_root=custom_index_root,
    embedding_model=embedding_model,
    text_field='retrieval_text',
    batch_size=8,
    device=device,
    build_dense=True,
    build_bm25=True,
)
manifest

In [ ]:
from src.retrieval import RetrievalEngine

engine = RetrievalEngine(custom_index_root, device=device)
query = 'Bu ?zel dok?manda kira uyu?mazl??? hakk?nda ne var?'
results = engine.dense_search(query, top_k=5)
[(r.get('citation_label'), round(r.get('score', 0), 4)) for r in results]

## Outputs

- `data/processed/custom_corpus_v1.csv`
- `data/processed/custom_corpus_v1.jsonl`
- `reports/custom_ingestion_report_v1.json`
- `indexes/custom_v1/`

Next: run `22_custom_rag_ui_demo.ipynb` to ask questions over either official-law or custom documents.